In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model, Model
import matplotlib.pyplot as plt
import numpy as np
import os

# Load the full trained model (architecture + weights) saved at the end of training
model_path = '/content/drive/MyDrive/defense/save/ResNet50V2_Blur_FullModel.keras'
model = load_model(model_path)

model.summary()

In [ ]:
# Find the nested ResNet50V2 sub-model inside the classifier
base_model_name = 'resnet50v2'
base_model = model.get_layer(base_model_name)

# Last conv/activation layer of ResNet50V2 before global average pooling
last_conv_layer_name = 'post_relu'

print("Base model:", base_model.name)
print("Last conv layer:", base_model.get_layer(last_conv_layer_name).output.shape)

In [ ]:
# Class labels — must match the alphabetical order used by flow_from_directory
# (this is exactly the order Keras assigned during training: see test_datagen.class_indices)
class_labels = [
    'Acne',
    'Dermatophyte Infections Ringworms',
    'Eczema',
    'Hair Diseases',
    'Insecets Bite and Sting',
    'Nail Disease',
    'Pigment Disorder',
    'Skin Cancer',
    'Urticaria Skin Alergey',
    'Vascular Tumors',
    'Vasculitis',
    'Versicolor'
]

In [ ]:
# Build the Grad-CAM function.
#
# NOTE: because ResNet50V2 is a *nested* sub-model, trying to rebuild a
# Model(model.inputs, [nested_layer.output, model.output]) breaks in Keras 3
# (the nested graph node doesn't reconnect cleanly -> KeyError at call time).
#
# The reliable fix is to NOT ask Keras to rebuild the graph at all: instead,
# inside the GradientTape we manually (1) run the base model to get the conv
# feature map, tell the tape to watch it, then (2) manually replay the
# remaining head layers (GAP -> dense -> dense -> dense -> dense) on top of
# it. This reproduces model(img_array) exactly, but keeps the conv output
# as an explicit tensor we can take gradients with respect to.

# Everything in the classifier AFTER the base model, in order:
head_layers = model.layers[2:]  # [global_average_pooling2d, dense, dense_1, dense_2, dense_3]

def generate_grad_cam(img_array, class_idx):
    img_tensor = tf.convert_to_tensor(img_array)

    with tf.GradientTape() as tape:
        conv_outputs = base_model(img_tensor)      # (1, 7, 7, 2048)
        tape.watch(conv_outputs)

        x = conv_outputs
        for layer in head_layers:
            x = layer(x)
        predictions = x                            # (1, 12)

        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)[0]

    # Importance weights via global average pooling of the gradients
    weights = tf.reduce_mean(grads, axis=(0, 1))

    # Weighted sum of the activation maps
    cam = tf.reduce_sum(tf.multiply(weights, conv_outputs[0]), axis=-1)

    # ReLU to keep only features that positively influence the predicted class
    cam = tf.nn.relu(cam)

    # Resize heatmap to the input image size
    cam = tf.image.resize(cam[tf.newaxis, ..., tf.newaxis], (224, 224))

    # Min-max normalize to [0, 1]
    cam_min, cam_max = tf.reduce_min(cam), tf.reduce_max(cam)
    cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

    return cam.numpy()


In [ ]:
import random

test_folder = '/content/drive/MyDrive/defense/dataset_split/test'
class_folders = sorted(os.listdir(test_folder))  # alphabetical order

for class_idx, class_folder in enumerate(class_folders):

    folder_path = os.path.join(test_folder, class_folder)

    img_name = random.choice(
        os.listdir(folder_path)
    )

    img_path = os.path.join(folder_path, img_name)

    # Load & preprocess image
    img = image.load_img(img_path, target_size=(224, 224))

    img_array = image.img_to_array(img)

    img_array = np.expand_dims(
        img_array,
        axis=0
    ) / 255.0

    # Generate Grad-CAM
    cam = generate_grad_cam(
        img_array,
        class_idx
    )

    # Display results
    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f'{class_labels[class_idx]}\nOriginal Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(cam[0, ..., 0], cmap='jet', vmin=0, vmax=1)
    plt.title('Grad-CAM Heatmap')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(img)
    plt.imshow(cam[0, ..., 0], cmap='jet', alpha=0.5, vmin=0, vmax=1)
    plt.title('Overlay')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

save_path = "/content/drive/MyDrive/defense/GradCAM_Result.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

print("Saved to:", save_path)

In [ ]:
def show_grad_cam_for_prediction(img_path, true_class_name=None):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    preds = model.predict(img_array, verbose=0)[0]
    pred_idx = int(np.argmax(preds))
    pred_label = class_labels[pred_idx]
    confidence = preds[pred_idx]

    cam = generate_grad_cam(img_array, pred_idx)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    title = f'True: {true_class_name}\n' if true_class_name else ''
    title += f'Predicted: {pred_label} ({confidence:.2%})'
    plt.title(title)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img)
    plt.imshow(cam[0, ..., 0], cmap='jet', alpha=0.5, vmin=0, vmax=1)
    plt.title('Grad-CAM (predicted class)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Example usage:
# show_grad_cam_for_prediction('/content/drive/MyDrive/defense/dataset_split/test/Acne/some_image.jpg', 'Acne')

In [ ]:
import os

folder = '/content/drive/MyDrive/defense/dataset_split/test/Acne'

img_name = os.listdir(folder)[0]

show_grad_cam_for_prediction(
    os.path.join(folder, img_name),
    'Acne'
)